# NYC 311 AutoDDG: milestone notebook

This notebook is the presentation of the project. Run it from top to bottom. Every milestone ends with an **Outcome** section: the table or file that goes into the report.

**Question.** How well can an AutoDDG-style description, grounded in an Apache Spark profile, describe NYC 311 service requests and make that dataset easier to find than the short portal text?

**Data.**
- Fact table: [311 Service Requests from 2020 to Present](https://data.cityofnewyork.us/d/erm2-nwe9), id `erm2-nwe9`.
- Dimension table: [ZIP Code Tabulation Areas](https://data.cityofnewyork.us/d/35j5-n34v), id `35j5-n34v` (221 rows).

The default run is a **dated sample**, not the full archive. The profile records the window, and the generated text is required to say so.


## Install requirements

1. Use **Python 3.11** and **JDK 17**. PySpark will not start on a machine with no `java` on `PATH`.
2. From the repo root: `python3.11 -m venv .venv && source .venv/bin/activate && pip install -r requirements.txt`.
3. `cp .env.example .env` and configure your LLM settings:
   - To use **Google Gemini API** (recommended, fast & free tier): set `LLM_PROVIDER=gemini` and `GEMINI_API_KEY`.
   - To use **OpenAI**: set `LLM_PROVIDER=openai` and `OPENAI_API_KEY`.
   - To use a **Local LLM** (e.g. Ollama): set `LLM_PROVIDER=local` and `LOCAL_LLM_BASE_URL`.
   Milestones 1, 2 ETL, and the human BM25 index run without an LLM key. Description generation and the judge require the chosen LLM.
4. Leave the default window (`2025-06-01` inclusive to `2025-06-03` exclusive, at most 20,000 rows) until this notebook finishes once.
5. Start Jupyter with the repo root as the working directory, or start it from `notebooks/`. The setup cell handles both.

Do not commit `.env`, the CSV pages, or the Parquet output.


In [3]:
!pip install -r ../requirements.txt


[notice] A new release of pip is available: 26.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


## How these steps map onto the proposal

The proposal's four milestones are the grading structure. This notebook uses the same four numbers.

| Proposal milestone | What this notebook finishes |
| --- | --- |
| 1. Setup, baselines, Spark | NYC Open Data lake, raw audit, and one Auctus table with a human description |
| 2. Generation and Spark patterns | ZIP/ZCTA join, profiles, AutoDDG SFD/UFD for 311 and Auctus, and OpenAI scores against the human text |
| 3. Evaluation | BM25 MRR and Recall@1 for portal text versus the 311 SFD |
| 4. Report, due 9 December | ACM-style paper from the outcome files, plus the GitHub link |

Stretch work from the proposal (a dataset with no description, a second judge, multimodal data) stays after this notebook.


## Setup check


In [6]:
import shutil
import sys
from pathlib import Path
import os

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from IPython.display import Markdown, display
from src import config


def show_table(title, rows, columns=None, limit=20):
    display(Markdown(f"### {title}"))
    rows = list(rows)[:limit]
    if not rows:
        display(Markdown("_No rows._"))
        return
    columns = columns or list(rows[0].keys())
    lines = [
        "| " + " | ".join(columns) + " |",
        "| " + " | ".join("---" for _ in columns) + " |",
    ]
    for row in rows:
        cells = [str(row.get(column, "")).replace("|", "/") for column in columns]
        lines.append("| " + " | ".join(cells) + " |")
    display(Markdown("\n".join(lines)))


java = shutil.which("java")
llm_status = (
    f"Gemini Key: {'LOADED' if os.getenv('GEMINI_API_KEY') or os.getenv('GOOGLE_API_KEY') else 'MISSING'}"
    if config.LLM_PROVIDER == "gemini"
    else (
        f"OpenAI Key: {'LOADED' if os.getenv('OPENAI_API_KEY') else 'MISSING'}"
        if config.LLM_PROVIDER == "openai"
        else f"Local Endpoint: {config.LOCAL_LLM_BASE_URL}"
    )
)

display(Markdown(
    f"""### Environment
| Item | Value |
| --- | --- |
| Repo | `{config.ROOT}` |
| Sample window | `{config.START_DATE}` inclusive, `{config.END_DATE}` exclusive |
| Row cap | `{config.SAMPLE_ROWS}` (`0` means the whole window) |
| Java on PATH | `{java or "MISSING — install Temurin 17 before Milestone 1 Spark"}` |
| LLM Provider | `{config.LLM_PROVIDER}` |
| Active Model | `{config.get_active_model()}` |
| LLM Auth / Status | `{llm_status}` |
"""
))


### Environment
| Item | Value |
| --- | --- |
| Repo | `/Users/lvcaitong/Desktop/NYU/Big data/Big Data final` |
| Sample window | `2025-06-01` inclusive, `2025-06-03` exclusive |
| Row cap | `20000` (`0` means the whole window) |
| Java on PATH | `/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home/bin/java` |
| LLM Provider | `gemini` |
| Active Model | `gemini-1.5-flash` |
| LLM Auth / Status | `Gemini Key: LOADED` |


# Milestone 1 — Ingestion, storage, and raw profiling

**Goal.** Land a reproducible slice of both the 311 and ZCTA tables (along with the catalog baseline) and record the schema, key uniqueness, and missingness that later stages are allowed to talk about.




## 1. Create the lake directories

**Goal:** Downloads raw project datasets and catalog metadata, then prints two quick summary tables to verify everything came in properly.

- Downloads the NYC 311 service request rows (for the above configured date window and row cap) from [311 Service Requests from 2020 to Present](https://data.cityofnewyork.us/Social-Services/311-Service-Requests-from-2020-to-Present/erm2-nwe9/about_data) (`erm2-nwe9`) into `data/raw/fact_311/` and returns `dataset_id`, `rows`, `pages`, and `where` to confirm filters and row counts.

- Downloads the ZCTA reference rows from [ZIP Code Tabulation Areas](https://data.cityofnewyork.us/City-Government/ZIP-Code-Tabulation-Areas/35j5-n34v/about_data) (`35j5-n34v`) into `data/raw/dim_zcta/` and returns a summary dictionary.

- Queries open-data catalog for `id`, `name`, `columns`, and a `180-character preview` to have a human-written baseline before generating descriptions with an LLM. The four other datasets are hard-coded in `src/config.py`. Those are popular NYC Open Data tables that already have human descriptions so the search/ranking test at later stages has other realistic NYC datasets to compare against.

In [7]:
from src.ingest import download_fact_311, download_zcta, fetch_catalog

fact_manifest = download_fact_311()
zcta_manifest = download_zcta()
catalog = fetch_catalog()

show_table(
    "Download manifest",
    [
        {"table": "311 fact", **{k: fact_manifest[k] for k in ("dataset_id", "rows", "pages", "where")}},
        {"table": "ZCTA dimension", **{k: zcta_manifest[k] for k in ("dataset_id", "rows", "pages", "select")}},
    ],
)
show_table(
    "Portal catalog (human baselines)",
    [
        {
            "id": row["id"],
            "name": row["name"],
            "columns": row["n_columns"],
            "description": (row["human_description"][:180] + "…") if len(row["human_description"]) > 180 else row["human_description"],
        }
        for row in catalog
    ],
)


### Download manifest

| table | dataset_id | rows | pages | where |
| --- | --- | --- | --- | --- |
| 311 fact | erm2-nwe9 | 19523 | 1 | created_date >= '2025-06-01T00:00:00.000' AND created_date < '2025-06-03T00:00:00.000' |
| ZCTA dimension | 35j5-n34v | 221 | 1 |  |

### Portal catalog (human baselines)

| id | name | columns | description |
| --- | --- | --- | --- |
| erm2-nwe9 | 311 Service Requests from 2020 to Present | 48 | NOTE: Learn more about the latest changes to this dataset: https://opendata.cityofnewyork.us/311-service-requests-from-2010-to-present-updates/ 311 responds to thousands of inquiri… |
| h9gi-nx95 | Motor Vehicle Collisions - Crashes | 29 | The Motor Vehicle Collisions crash table contains details on the crash event. Each row represents a crash event. The Motor Vehicle Collisions data tables contain information from a… |
| 43nn-pn8j | DOHMH New York City Restaurant Inspection Results | 31 | The dataset contains every sustained or not yet adjudicated violation citation from every full or special program inspection conducted up to three years prior to the most recent in… |
| 5uac-w243 | NYPD Complaint Data Current (Year To Date) | 41 | This dataset includes all valid felony, misdemeanor, and violation crimes reported to the New York City Police Department (NYPD) for all complete quarters so far this year (2019). … |
| 35j5-n34v | ZIP Code Tabulation Areas | 8 | This is a subset of ZIP Code Tabulation Areas or ZCTAs (pronounced zik-tahs) that fall within New York City. ZCTAs are a geographic product of the U.S. Census Bureau created to all… |
| datamart.socrata.data-cdc-gov.kee5-23sr | PLACES: ZCTA Data (GIS Friendly Format), 2024 release | 84 | This dataset contains model-based ZIP Code Tabulation Area (ZCTA) level estimates in GIS-friendly format. PLACES covers the entire United States—50 states and the District of Colum… |

## 2. Adds an external joinable table for discovery & enrichment from Auctus

**Goal:** Enrich 311 data on ZIP code from external Auctus dataset

Fetch ZIP data from [PLACES: ZCTA Data (GIS Friendly Format), 2024 release](https://catalog.data.gov/dataset/places-zcta-data-gis-friendly-format-2024-release) and record it in the catalog so it can later connect 311 service requests to neighborhood health data via ZIP code.

Downloads the raw CSV to `data/raw/auctus_places_zcta/part-00001.csv` and appends its metadata/description to `data/processed/catalog.json`.



In [5]:
from src.ingest import download_auctus_places

auctus_record = download_auctus_places()
show_table(
    "Outcome — Auctus catalog entry",
    [{
        "id": auctus_record["id"],
        "name": auctus_record["name"],
        "n_columns": auctus_record["n_columns"],
        "join": "clean_zip = ZCTA5",
        "description": auctus_record["human_description"][:180] + "…",
    }],
)


### Outcome — Auctus catalog entry

| id | name | n_columns | join | description |
| --- | --- | --- | --- | --- |
| datamart.socrata.data-cdc-gov.kee5-23sr | PLACES: ZCTA Data (GIS Friendly Format), 2024 release | 84 | clean_zip = ZCTA5 | This dataset contains model-based ZIP Code Tabulation Area (ZCTA) level estimates in GIS-friendly format. PLACES covers the entire United States—50 states and the District of Colum… |

## 3. Spark load and structural audit

**Goal:** Local Apache Spark load and structural audit. To catch data-quality issues early, protect string formatting and create the `m1_raw_audit.json` artifact that documents the sample window and column roles for later milestones and report outputs.

<u>Code explanation:</u>
- Checks whether a `java` binary is discoverable on `PATH` before launching PySpark. If missing, install it, export `JAVA_HOME`, restart the kernel, and run from the setup cell again.
- Calls `spark = start_spark()` to initialize local Spark session.
- `fact_raw = load_fact(spark)` reads the raw 311 CSV pages with schema inference disabled (reading columns as strings so values like `incident_zip` keep leading zeros or formatting intact).
- `zcta_raw = load_zcta(spark)` loads the ZCTA dimension CSV into Spark.
- `audit_raw(fact_raw)` computes structural diagnostics on fact_raw and writes `data/processed/m1_raw_audit.json`

<u>Table explanation:</u>
1. `Outcome - raw 311 audit` shows a compact two-column check table. Confirm the total row count and the distinct `unique_key` count. Also confirm the 311 table loaded all expected columns and the ZCTA table loaded its 221 reference rows.
  
2. `Null or blank counts on fields the join and the description depend on` lists each audited field alongside its `null_or_blank` count to know where missing or blank values could affect joins or AutoDDG descriptions. Missing or blank ZIP codes won't match ZCTA records (ZCTA5), which tells us how many rows might drop or stay unmatched in a join.
 
3. `Column roles` groups column names by role and prints a comma-joined list per role. Downstream profiling and description generation rely on column roles to pick representative fields—for example, using temporal columns for date coverage, spatial columns for geography/ZIPs, and categorical/text columns for complaint types or summaries. If an ID or free-text column gets misclassified as categorical (or vice versa), it can distort summary stats or clutter generated descriptions.


In [8]:
from src.config import RAW_311

print(RAW_311)

parts = sorted(str(path.resolve().as_uri()) for path in RAW_311.glob("part-*.csv"))
print(parts)

/Users/lvcaitong/Desktop/NYU/Big data/Big Data final/data/raw/fact_311
['file:///Users/lvcaitong/Desktop/NYU/Big%20data/Big%20Data%20final/data/raw/fact_311/part-00001.csv']


In [9]:
if shutil.which("java") is None:
    raise RuntimeError(
        "Java is not on PATH. Install Temurin 17, export JAVA_HOME, restart the kernel, and rerun from the setup cell."
    )

from src.spark_etl import audit_raw, load_fact, load_zcta, start_spark

spark = start_spark()
fact_raw = load_fact(spark)
zcta_raw = load_zcta(spark)
audit = audit_raw(fact_raw)

display(Markdown(
    f"""### Outcome — raw 311 audit
| Check | Value |
| --- | --- |
| Rows in this sample | {audit['total_rows']:,} |
| Distinct `unique_key` | {audit['unique_keys']:,} |
| Duplicate keys | {audit['duplicate_keys']:,} |
| Columns | {audit['n_columns']} |
| ZCTA rows loaded | {zcta_raw.count():,} |
| Window | {audit['sample_window']['start']} to {audit['sample_window']['end']} (end exclusive) |
"""
))
show_table(
    "Null or blank counts on fields the join and the description depend on",
    [{"column": name, "null_or_blank": n} for name, n in audit["null_or_blank_counts"].items()],
)
roles = {}
for column in audit["columns"]:
    roles.setdefault(column["role"], []).append(column["name"])
show_table(
    "Column roles",
    [{"role": role, "columns": ", ".join(names)} for role, names in sorted(roles.items())],
)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/23 11:14:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Outcome — raw 311 audit
| Check | Value |
| --- | --- |
| Rows in this sample | 19,523 |
| Distinct `unique_key` | 19,523 |
| Duplicate keys | 0 |
| Columns | 44 |
| ZCTA rows loaded | 221 |
| Window | 2025-06-01 to 2025-06-03 (end exclusive) |


### Null or blank counts on fields the join and the description depend on

| column | null_or_blank |
| --- | --- |
| unique_key | 0 |
| created_date | 0 |
| closed_date | 230 |
| complaint_type | 0 |
| descriptor | 0 |
| incident_zip | 176 |
| borough | 0 |
| latitude | 331 |
| longitude | 331 |
| status | 0 |

### Column roles

| role | columns |
| --- | --- |
| categorical | agency, agency_name, complaint_type, location_type, street_name, cross_street_1, cross_street_2, intersection_street_1, intersection_street_2, address_type, landmark, facility_type, status, community_board, council_district, police_precinct, open_data_channel_type, park_facility_name, park_borough, vehicle_type, taxi_company_borough, taxi_pick_up_location, bridge_highway_name, bridge_highway_direction, road_ramp, bridge_highway_segment |
| identifier | unique_key, bbl |
| spatial | incident_zip, city, borough, x_coordinate_state_plane, y_coordinate_state_plane, latitude, longitude, location |
| temporal | created_date, closed_date, due_date, resolution_action_updated_date |
| text | descriptor, descriptor_2, incident_address, resolution_description |

## 4. What milestone 1 completed

<u> Data Ingestion:</u>
- Downloaded the sampled `311 Service Requests from 2020 to Present` table in paginated chunks within the default date window and row cap.
- Fetched the `ZIP Code Tabulation Areas` reference table.
- Pulled the external `CDC PLACES 2024 ZCTA` dataset via Auctus to capture human baseline metadata.

<u>Data audit with Spark:</u>
- Verified Java on PATH, started a local Spark session
- Read columns as strings to preserve formatting (such as leading zeros or ZIP strings)
- Audited row counts, unique_key uniqueness (duplicate_keys = 0), missing/blank counts on key fields (incident_zip, created_date, complaint_type), and semantic column roles (categorical, identifier, spatial, temporal, text).

<u>Outcome files & artifacts:</u>
- data/raw/fact_311/ — Raw paginated NYC 311 CSV data and download manifest (0 < rows <= SAMPLE_ROWS).
- data/raw/dim_zcta/ — Raw ZCTA reference CSV data and manifest.
- data/raw/auctus_places_zcta/part-00001.csv — Raw CDC PLACES ZCTA CSV for joining on clean_zip = ZCTA5.
- data/processed/catalog.json — Dataset catalog containing human baseline descriptions for search/retrieval and LLM evaluation.
- data/processed/m1_raw_audit.json — Raw Spark audit summary recording row/key counts, sample window bounds, missingness ground truth, and column role classifications.


# Milestone 2 — Cleaning, generation, and baseline scoring

**Goal.** Produce the enriched 311 table, generate AutoDDG descriptions for that table and for the Auctus PLACES table, and score both generated texts and the human-authored descriptions on faithfulness, completeness, and searchability.

**Do these steps**

1. Build `clean_zip` from the digits in `incident_zip`. Keep the first five. ZIP+4 values such as `10001-1234` become `10001`.
2. Parse `created_date` and `closed_date` with both timestamp styles. The SODA file is `yyyy-MM-dd'T'HH:mm:ss.SSS`. A bulk portal export is `MM/dd/yyyy hh:mm:ss a`.
3. Drop rows whose ZIP is not five digits or whose created timestamp did not parse. Record how many rows each rule removes.
4. Broadcast the ZCTA table and left-join on `clean_zip = dim_zip`. Report the match rate. A USPS ZIP and a Census ZCTA are not the same geography.
5. Compute median hours to close for the top complaint types, then write Parquet partitioned by `year` and `month`.
6. Build a Spark profile for the enriched 311 sample and for the Auctus PLACES ZCTA table. The profile is the only evidence the model may cite.
7. Generate an SFD and a UFD for each profile.
8. Run the same judge on the generated text and on the human portal description. The profile is the ground truth for both scores.

The judge compares each text with the profile of the table you actually analyzed. The 311 portal description covers the full archive, so it can score lower on faithfulness for this two-day sample.


In [10]:
from src.spark_etl import (
    apply_fact_filters,
    broadcast_join,
    clean_fact,
    clean_zcta,
    join_report,
    partition_counts,
    resolution_pattern,
    write_curated,
)

fact_clean = clean_fact(fact_raw)
fact_kept, cleaning = apply_fact_filters(fact_clean)
zcta_clean = clean_zcta(zcta_raw)
joined = broadcast_join(fact_kept, zcta_clean)
joined.cache()
match = join_report(joined)
resolution = resolution_pattern(joined)
curated_path = write_curated(joined)
partitions = partition_counts(spark)

show_table("Outcome — rows removed by cleaning", [cleaning])
display(Markdown(
    f"""### Outcome — broadcast join
| Check | Value |
| --- | --- |
| Rows after cleaning | {match['rows']:,} |
| Matched to a ZCTA | {match['matched_rows']:,} |
| Unmatched | {match['unmatched_rows']:,} |
| Match rate | {match['match_rate']:.1%} |
| Curated Parquet | `{curated_path}` |
"""
))
display(Markdown(match["note"]))
show_table("Most common ZIPs with no ZCTA match", match["top_unmatched_zips"])
show_table("Outcome — median hours to close, top complaint types", resolution)
show_table("Outcome — Parquet partitions", partitions)


26/11/23 12:13:12 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

### Outcome — rows removed by cleaning

| rows_before_filter | rows_with_valid_zip | rows_dropped_bad_zip | rows_dropped_bad_timestamp_after_zip | rows_kept |
| --- | --- | --- | --- | --- |
| 19523 | 19347 | 176 | 0 | 19347 |

### Outcome — broadcast join
| Check | Value |
| --- | --- |
| Rows after cleaning | 19,347 |
| Matched to a ZCTA | 19,335 |
| Unmatched | 12 |
| Match rate | 99.9% |
| Curated Parquet | `/Users/lvcaitong/Desktop/NYU/Big data/Big Data final/data/curated/nyc_311_zcta_enriched` |


incident_zip is a USPS ZIP and zcta5 is a Census ZCTA. Equality is an approximation, so a match rate below 1 is expected.

### Most common ZIPs with no ZCTA match

| clean_zip | rows |
| --- | --- |
| 10121 | 4 |
| 10118 | 3 |
| 10000 | 1 |
| 07032 | 1 |
| 10048 | 1 |
| 21044 | 1 |
| 10281 | 1 |

### Outcome — median hours to close, top complaint types

| complaint_type | n | median_resolution_hours |
| --- | --- | --- |
| Illegal Parking | 3329 | 1.2 |
| Noise - Residential | 2427 | 0.71 |
| Noise - Street/Sidewalk | 1682 | 0.58 |
| Blocked Driveway | 862 | 1.17 |
| UNSANITARY CONDITION | 650 | 208.8 |
| Noise - Commercial | 571 | 0.51 |
| Street Condition | 417 | 26.21 |
| Noise - Vehicle | 409 | 0.73 |
| Dirty Condition | 402 | 21.34 |
| HEAT/HOT WATER | 360 | 32.68 |

### Outcome — Parquet partitions

| year | month | rows |
| --- | --- | --- |
| 2025 | 6 | 19347 |

### AutoDDG descriptions

`build_profile` summarizes the enriched 311 sample. `build_auctus_profile` summarizes the full PLACES ZCTA table. `generate_descriptions` writes an SFD and a UFD from that profile using the configured LLM (`gemini`, `local`, or `openai`).


In [12]:
from src.autoddg_engine import generate_descriptions
from src.config import PROCESSED
from src.profile import build_profile

profile = build_profile(
    joined,
    match,
    resolution,
    is_sample=config.SAMPLE_ROWS != 0,
)
descriptions = generate_descriptions(
    profile,
    provider=LLM_PROVIDER,
    model=ACTIVE_MODEL,
    output_path=PROCESSED / "autoddg_descriptions.json",
)
display(Markdown("### 311 search-focused description\n\n" + descriptions["sfd"]))
display(Markdown("### 311 user-focused description\n\n" + descriptions["ufd"]))


InternalServerError: Error code: 503 - [{'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}]

In [ ]:
from src.profile import build_auctus_profile, load_auctus_frame

auctus_frame = load_auctus_frame(spark)
auctus_profile = build_auctus_profile(auctus_frame)
auctus_descriptions = generate_descriptions(
    auctus_profile,
    provider=LLM_PROVIDER,
    model=ACTIVE_MODEL,
    output_path=PROCESSED / "autoddg_descriptions_auctus.json",
)
display(Markdown("### Auctus search-focused description\n\n" + auctus_descriptions["sfd"]))
display(Markdown("### Auctus user-focused description\n\n" + auctus_descriptions["ufd"]))


In [ ]:
from src.autoddg_engine import human_description, score_against_baseline, write_scorecard
from src.config import AUCTUS_ID, FACT_ID

scorecard = write_scorecard([
    score_against_baseline(
        profile,
        descriptions,
        human_description(FACT_ID),
        "311",
        provider=LLM_PROVIDER,
        model=ACTIVE_MODEL,
    ),
    score_against_baseline(
        auctus_profile,
        auctus_descriptions,
        human_description(AUCTUS_ID),
        "auctus",
        provider=LLM_PROVIDER,
        model=ACTIVE_MODEL,
    ),
])
show_table(
    "Outcome — baseline scores against the Spark profile",
    scorecard["rows"],
    ["dataset", "text", "faithfulness", "completeness", "searchability"],
)
display(Markdown("Saved to `data/processed/eval_judge_comparison.json`."))


### Milestone 2 is done when

- The cleaning report, join match rate, and Parquet partition count are saved.
- `content_profile.json` and `content_profile_auctus.json` exist.
- Both description files exist, and every number in them appears in the matching profile.
- `eval_judge_comparison.json` has four rows: generated and human for 311, and generated and human for the Auctus table.


## Milestone 3 — Retrieval benchmark

Description generation and the OpenAI baseline scores are in Milestone 2. This milestone asks whether the 311 SFD is easier to retrieve than the portal description.

One document per catalog dataset. Queries stay frozen in `src/config.py`. Report MRR and Recall@1. The second index replaces only the 311 document with its SFD.


### BM25

Do not compare one raw BM25 score on a short portal sentence with one score on a long SFD. Index one document per catalog dataset, keep the queries in `src/config.py` frozen, and report MRR and Recall@1. The second index replaces only the 311 document with its SFD.


In [ ]:
from src.search_eval import evaluate_retrieval

retrieval = evaluate_retrieval()
human = retrieval["human_portal_index"]
sfd_index = retrieval["autoddg_sfd_index"]
display(Markdown(
    f"""### Outcome — retrieval
| Index | Documents | Queries | MRR | Recall@1 |
| --- | --- | --- | --- | --- |
| Portal descriptions | {human['n_documents']} | {human['n_queries']} | {human['mrr']} | {human['recall_at_1']} |
| 311 document replaced by SFD | {sfd_index['n_documents'] if sfd_index else "—"} | {sfd_index['n_queries'] if sfd_index else "—"} | {sfd_index['mrr'] if sfd_index else "run Milestone 3"} | {sfd_index['recall_at_1'] if sfd_index else "run Milestone 3"} |
"""
))
show_table(
    "Rank of the correct dataset under portal text",
    human["queries"],
    ["query", "relevant_id", "rank", "top_id"],
)
if sfd_index:
    show_table(
        "Rank of the correct dataset after the 311 SFD swap",
        sfd_index["queries"],
        ["query", "relevant_id", "rank", "top_id"],
    )
display(Markdown("Saved to `data/processed/eval_bm25.json`."))


## Milestone 4 — Report

The faithfulness, completeness, and searchability comparison is already in Milestone 2 (`data/processed/eval_judge_comparison.json`). Use that table, the two description files, the join match rate, and the BM25 table in the paper.

### What the proposal still requires

1. **Missing dictionary.** Profile one NYC Open Data or Auctus dataset whose description is empty, and generate an SFD/UFD for it. Keep those numbers separate from the 311 and PLACES scores.
2. **Scale.** Widen `START_DATE` / `END_DATE` and set `SAMPLE_ROWS=0` only after this sample run is the one you intend to cite.
3. **Second judge and multimodal work.** Only after the items above.
4. Publish the repo and put its URL in the ACM-style report. Map the outcome files into Introduction, Problem formulation, Related work, Methods, Results, and References.
